# 11｜電輔車借出量對帳
依檔案借出時間限定一月，包含所有日型與時段。先從已知起站旅次分站計數，再加回未歸站量，逐日對照原始借出量。

In [1]:
from zipfile import ZipFile
import pandas as pd

with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as archive:
    with archive.open('202501-capitalbikeshare-tripdata.csv') as source:
        rides = pd.read_csv(source, usecols=['rideable_type', 'started_at', 'start_station_id'], dtype='string')
started = pd.to_datetime(rides.started_at, format='mixed', errors='raise')
keep = rides.rideable_type.eq('electric_bike') & started.ge('2025-01-01') & started.lt('2025-02-01')
borrow = rides.loc[keep].assign(date=started[keep].dt.normalize())
borrow['station_id'] = borrow.start_station_id.fillna('').str.strip()
known = borrow.loc[borrow.station_id.ne('')].groupby(['date','station_id']).size()
unknown = borrow.loc[borrow.station_id.eq('')].groupby('date').size()
daily = pd.concat([borrow.groupby('date').size().rename('total'),
                   known.groupby(level='date').sum().rename('known'),
                   unknown.rename('unknown')], axis=1).fillna(0).astype(int)
assert len(daily) == 31
assert daily.total.eq(daily.known + daily.unknown).all()
assert daily.sum().tolist() == [206753,118643,88110]
daily['unknown_pct'] = (daily.unknown / daily.total * 100).round(2)
print(daily.to_string())
print('January totals:', daily[['total','known','unknown']].sum().to_dict())


            total  known  unknown  unknown_pct
date                                          
2025-01-01   6706   3584     3122        46.56
2025-01-02   7877   4330     3547        45.03
2025-01-03   6994   3953     3041        43.48
2025-01-04   6063   3388     2675        44.12
2025-01-05   6079   3401     2678        44.05
2025-01-06   1516    568      948        62.53
2025-01-07   2679   1181     1498        55.92
2025-01-08   4377   2182     2195        50.15
2025-01-09   4815   2379     2436        50.59
2025-01-10   5844   2991     2853        48.82
2025-01-11   4840   2567     2273        46.96
2025-01-12   5805   3342     2463        42.43
2025-01-13   7772   4362     3410        43.88
2025-01-14   8222   4890     3332        40.53
2025-01-15   7730   4600     3130        40.49
2025-01-16   7610   4441     3169        41.64
2025-01-17   8285   4991     3294        39.76
2025-01-18   7455   4522     2933        39.34
2025-01-19   3989   2271     1718        43.07
2025-01-20   